# Silver Layer - Sales Transactions

Transform raw Bronze data into clean Silver data.

**Source:** `end-to-end_pipeline.bronze.sales_transactions`  
**Target:** `end-to-end_pipeline.silver.sales_transactions`

**Approach:** "Profile → Inspect → Transform → Validate"

## Step 1: Profile Bronze Data

Inspect data quality issues before transformation:

* Duplicate transaction_ids
* NULL customer_ids (guest purchases are valid)
* Invalid quantities (≤ 0)
* Invalid prices or costs
* Discount issues (NULL, negative, or > 100%)
* Inconsistent order_status values
* Date range coverage

This single query replaces 9+ separate checks.

In [0]:
%sql
-- CELL 1: Profile Bronze sales_transactions

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_ids,
    COUNT(*) - COUNT(DISTINCT transaction_id) AS duplicate_transactions,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_ids,

    SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantities,

    SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END) AS invalid_unit_prices,

    SUM(CASE WHEN unit_cost < 0 THEN 1 ELSE 0 END) AS invalid_unit_costs,

    SUM(
        CASE 
            WHEN discount_pct IS NULL 
              OR discount_pct < 0 
              OR discount_pct > 1
            THEN 1 ELSE 0
        END
    ) AS discount_issues,

    COUNT(DISTINCT order_status) AS distinct_order_statuses,

    MIN(order_date) AS min_raw_date,
    MAX(order_date) AS max_raw_date

FROM `end-to-end_pipeline`.bronze.sales_transactions;

total_rows,distinct_transaction_ids,duplicate_transactions,null_customer_ids,invalid_quantities,invalid_unit_prices,invalid_unit_costs,discount_issues,distinct_order_statuses,min_raw_date,max_raw_date
25005,25000,5,3,1,93,0,2,6,03/15/2024,2025-12-31


## Step 2: Transform to Silver

Apply all data quality fixes in one pass:

**Data Cleaning:**
* TRIM whitespace from all identifiers
* Replace NULL customer_id with 'UNKNOWN'
* Standardize date formats (yyyy-MM-dd, MM/dd/yyyy, dd-MM-yyyy)

**Data Type Enforcement:**
* CAST quantity to INT
* CAST prices to DECIMAL(10,2)
* COALESCE NULL discount_pct to 0

**Standardization:**
* Normalize order_status (COMPLETED/COMPLETE → Completed)
* INITCAP payment_method for consistent capitalization

**Deduplication:**
* ROW_NUMBER() to keep first occurrence per transaction_id

**Quality Filters:**
* Remove invalid quantities (≤ 0), prices (≤ 0), costs (< 0), discounts (< 0 or > 1)
* Remove NULL transaction_id or order_date

In [0]:
%sql
-- CELL 2: Create cleaned Silver sales_transactions table

CREATE OR REPLACE TABLE `end-to-end_pipeline`.silver.sales_transactions AS

WITH cleaned AS (

    SELECT

        -- Clean identifiers
        TRIM(transaction_id) AS transaction_id,
        TRIM(order_id) AS order_id,

        -- Standardize date formats
        COALESCE(
            TRY_TO_DATE(order_date, 'yyyy-MM-dd'),
            TRY_TO_DATE(order_date, 'MM/dd/yyyy'),
            TRY_TO_DATE(order_date, 'dd-MM-yyyy')
        ) AS order_date,

        -- Preserve valid transactions even when customer is unknown
        COALESCE(TRIM(customer_id), 'UNKNOWN') AS customer_id,

        TRIM(product_id) AS product_id,
        TRIM(store_id) AS store_id,

        -- Correct numeric data types
        CAST(quantity AS INT) AS quantity,
        CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
        CAST(unit_cost AS DECIMAL(10,2)) AS unit_cost,

        -- Missing discount = no discount
        COALESCE(
            CAST(discount_pct AS DECIMAL(5,2)),
            0
        ) AS discount_pct,

        -- Standardize status
        CASE
            WHEN UPPER(TRIM(order_status)) IN ('COMPLETED', 'COMPLETE')
                THEN 'Completed'

            WHEN UPPER(TRIM(order_status)) IN ('CANCELLED', 'CANCELED')
                THEN 'Cancelled'

            WHEN UPPER(TRIM(order_status)) = 'RETURNED'
                THEN 'Returned'

            ELSE INITCAP(TRIM(order_status))
        END AS order_status,

        -- Standardize payment method
        INITCAP(TRIM(payment_method)) AS payment_method,

        -- Identify true duplicate transaction IDs
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(transaction_id)
            ORDER BY order_id
        ) AS row_num

    FROM `end-to-end_pipeline`.bronze.sales_transactions
)

SELECT
    transaction_id,
    order_id,
    order_date,
    customer_id,
    product_id,
    store_id,
    quantity,
    unit_price,
    unit_cost,
    discount_pct,
    order_status,
    payment_method

FROM cleaned

WHERE row_num = 1
  AND transaction_id IS NOT NULL
  AND order_date IS NOT NULL
  AND quantity > 0
  AND unit_price > 0
  AND unit_cost >= 0
  AND discount_pct BETWEEN 0 AND 1;

num_affected_rows,num_inserted_rows


## Step 3: Validate Silver Data

Verify all transformations were successful.

**Expected Results (all should be zero):**
* remaining_duplicates = 0
* null_customer_ids = 0
* invalid_quantities = 0
* invalid_unit_prices = 0
* invalid_unit_costs = 0
* invalid_discounts = 0
* Date range is clean and valid

If any metric > 0, the transformation has an issue.

In [0]:
%sql
-- CELL 3: Validate Silver sales_transactions

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT transaction_id) AS distinct_transaction_ids,

    COUNT(*) - COUNT(DISTINCT transaction_id) 
        AS remaining_duplicates,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
        AS null_customer_ids,

    SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END)
        AS invalid_quantities,

    SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END)
        AS invalid_unit_prices,

    SUM(CASE WHEN unit_cost < 0 THEN 1 ELSE 0 END)
        AS invalid_unit_costs,

    SUM(
        CASE 
            WHEN discount_pct < 0 OR discount_pct > 1
            THEN 1 ELSE 0
        END
    ) AS invalid_discounts,

    MIN(order_date) AS earliest_date,
    MAX(order_date) AS latest_date

FROM `end-to-end_pipeline`.silver.sales_transactions;

total_rows,distinct_transaction_ids,remaining_duplicates,null_customer_ids,invalid_quantities,invalid_unit_prices,invalid_unit_costs,invalid_discounts,earliest_date,latest_date
24905,24905,0,0,0,0,0,0,2021-01-01,2025-12-31
